<a href="https://colab.research.google.com/github/JeremiahSents/Knowledge-Engineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical Exercise 3: Searching Agent — Self-Driving Trailer

Find the cheapest route from Nairobi to Kisumu using Uniform Cost Search (path cost = distance in km).

In [ ]:
import heapq
#This dictionary represents the graph we made on the routes to Kisumu. The key is the town and the values represent all
#other towns that it can go to
graph = {
    "Nairobi": {"Thika": 45 , "Narok": 142 , "Naivasha": 90},
    "Thika": {"Nyeri": 105},
    "Nyeri": {"Nanyuki": 60, "Nyahururu": 100},
    "Nanyuki": {},
    "Nyahururu": {"Nakuru": 70},
    "Naivasha": {"Nakuru": 70},
    "Nakuru": {"Kericho": 107, "Eldoret": 156},
    "Narok": {"Kisii": 170},
    "Kericho": {"Kisii": 100, "Kisumu": 84},
    "Eldoret": {"Kakamega": 115, "Webuye": 60},
    "Webuye": {"Kakamega": 40},
    "Kisii": {"Kisumu": 112},
    "Kakamega":{"Kisumu": 52},
}

In [ ]:
#this is where we will store the candidate towns and decide which path to take(the cheapest)
priority_queue = []
#this ensures that we select the cheapest path in the heap
heapq.heappush(priority_queue, (0, "Nairobi", ["Nairobi"]))
#this is a dictionary that stores the cheapest path to a specific town from the start town(Nairobi)
best_cost = {
    "Nairobi": 0,
}

#This loops through all the available paths to Kisumu and picks the cheapest one
while priority_queue:
    # 1. Pop the cheapest entry
    current_cost,current_town,current_path = (heapq.heappop(priority_queue))
    # 2. Skip outdated entries
    if current_cost > best_cost[current_town]:
      continue
    # 3. Check if current town is Kisumu
    if current_town == "Kisumu":
      print(f"Minimum cost: {best_cost[current_town]}")
      print(f"Path: {current_path}")
      break
    # 4. Explore neighbouring towns
    for town, cost in graph[current_town].items():
      # 5. Calculate new costs
      new_cost = cost + current_cost
      if town not in best_cost or new_cost < best_cost[town]:
        # 6. Update best_cost and priority_queue
        best_cost[town] = new_cost
        new_path = current_path + [town]
        heapq.heappush(priority_queue, (new_cost,town,new_path))

Minimum cost: 351
Path: ['Nairobi', 'Naivasha', 'Nakuru', 'Kericho', 'Kisumu']


# Knowledge Engineering — Self-Driving Trailer

Builds on practical exercise 3: the trailer's knowledge base is Kenya's highway network.

**Step 1: Identify the Task**

Give the self-driving trailer the knowledge it needs to plan a trip from Nairobi to Kisumu.

* **Input:** the trailer's current town
* **Output:** roads it can take next, their distances, and whether Kisumu can be reached
* **Goal:** only plan routes on roads that exist

**Step 2: Assemble Relevant Knowledge**

* Towns in the Nairobi, Central Highlands, Rift Valley and Lake Victoria Basin regions
* Direct highway links between towns (one-way, as in the Part 3 directed graph)
* Distance of each link in km

**Source:** Google Maps distances from Practical Exercise 3, entered by hand. No APIs or web scraping were used.

**Step 3: Define the Vocabulary**

* **Predicates:** `Town(t)`, `Road(a, b)`, `Distance(a, b, km)`, `Reachable(a, b)`
* **Constants:** Nairobi, Thika, Nyeri, Nanyuki, Nyahururu, Naivasha, Nakuru, Narok, Kericho, Eldoret, Webuye, Kisii, Kakamega, Kisumu

**Step 4: Encode General Knowledge about the Domain** `[3 marks]`

`roads` stores every `Road(a, b)` with its `Distance(a, b, km)`; the functions below encode each predicate.

In [ ]:
# Road(a, b) and Distance(a, b, km): town -> {next town: km}
roads = {
    "Nairobi":   {"Thika": 45, "Narok": 142, "Naivasha": 90},
    "Thika":     {"Nyeri": 105},
    "Nyeri":     {"Nanyuki": 60, "Nyahururu": 100},
    "Nanyuki":   {},
    "Nyahururu": {"Nakuru": 70},
    "Naivasha":  {"Nakuru": 70},
    "Nakuru":    {"Kericho": 107, "Eldoret": 156},
    "Narok":     {"Kisii": 170},
    "Kericho":   {"Kisii": 100, "Kisumu": 84},
    "Eldoret":   {"Kakamega": 115, "Webuye": 60},
    "Webuye":    {"Kakamega": 40},
    "Kisii":     {"Kisumu": 112},
    "Kakamega":  {"Kisumu": 52},
    "Kisumu":    {},
}

def Town(t):
    return t in roads

def Road(a, b):
    return Town(a) and b in roads[a]

def Distance(a, b):
    return roads[a][b] if Road(a, b) else None

def Reachable(a, b):
    # True if the trailer can drive from a to b along known roads.
    if not (Town(a) and Town(b)):
        return False
    visited, stack = set(), [a]
    while stack:
        town = stack.pop()
        if town == b:
            return True
        if town not in visited:
            visited.add(town)
            stack.extend(roads[town])
    return False

print("Towns in KB:", len(roads))
print("Road(Nakuru, Kericho):", Road("Nakuru", "Kericho"), "|", Distance("Nakuru", "Kericho"), "km")

Towns in KB: 14
Road(Nakuru, Kericho): True | 107 km


**Step 5: Add New Facts to the Knowledge Base** `[4 marks]`

In [ ]:
def add_town(town):
    # Add a town with no outgoing roads yet.
    town = town.strip().title()
    if town not in roads:
        roads[town] = {}
        print(f"Added town: {town}")
    return town

def add_road(a, b, km):
    # Add a one-way road a -> b. Unknown towns are added first.
    a, b = a.strip().title(), b.strip().title()
    if a == b:
        print("A road must connect two different towns.")
        return
    if not isinstance(km, (int, float)) or km <= 0:
        print("Distance must be a positive number.")
        return
    add_town(a)
    add_town(b)
    if b in roads[a]:
        print(f"Updated road: {a} -> {b}, {roads[a][b]} km -> {km} km")
    else:
        print(f"Added road: {a} -> {b} ({km} km)")
    roads[a][b] = km

def remove_road(a, b):
    # Remove a one-way road if it exists.
    if Road(a, b):
        del roads[a][b]
        print(f"Removed road: {a} -> {b}")
    else:
        print(f"No road {a} -> {b} to remove.")

def add_road_from_input():
    # Ask the user for a new road and add it to the knowledge base.
    a = input("From town: ")
    b = input("To town: ")
    try:
        km = float(input("Distance in km: "))
    except ValueError:
        print("Distance must be a number.")
        return
    add_road(a, b, km)

# Adding facts directly from Google Maps
add_road("Eldoret", "Kapsabet", 46.1)
add_road("Kapsabet", "Kisumu", 67.2)

# Invalid inputs are rejected
add_road("Nairobi", "Nairobi", 10)
add_road("Nairobi", "Thika", -5)

# View the updated KB
print("\nTowns in KB:", len(roads))
print("Road(Eldoret, Kapsabet):", Road("Eldoret", "Kapsabet"), "|",
      Distance("Eldoret", "Kapsabet"), "km")

Added town: Kapsabet
Added road: Eldoret -> Kapsabet (46.1 km)
Added road: Kapsabet -> Kisumu (67.2 km)
A road must connect two different towns.
Distance must be a positive number.

Towns in KB: 15
Road(Eldoret, Kapsabet): True | 46.1 km


**Input call, to demonstrate user input for step 5**

In [ ]:
# add_road_from_input()

**Steps 6 & 7: Encode Specific Problem Instance (Facts) & Pose Queries and Get Answers** `[3 marks]`

In [ ]:
import heapq

# Step 6: the specific problem instance (fixed values for faster reruns)
start = "Nairobi"
goal = "Kisumu"
# Takes user input
# start = input("Start town: ").strip().title()
# goal = input("Destination: ").strip().title()

# Step 7: pose queries and get answers
def find_route(start, goal):
    # Uniform cost search over `roads`.
    if not (Town(start) and Town(goal)):
        return None # there is no route
    queue = [(0, start, [start])]
    best_cost = {start: 0}
    while queue:
        cost, town, path = heapq.heappop(queue)
        if cost > best_cost[town]:
            continue
        if town == goal:
            return cost, path # there is a route
        for nxt, km in roads[town].items():
            new_cost = cost + km
            if nxt not in best_cost or new_cost < best_cost[nxt]:
                best_cost[nxt] = new_cost
                heapq.heappush(queue, (new_cost, nxt, path + [nxt]))
    return None

def plan_trip(start, goal):
    print("\n=== Trailer Route Planner ===")
    print(f"Trip: {start} -> {goal}")
    if not Town(start):
        print(f"Unknown town: {start}")
        return
    if not Town(goal):
        print(f"Unknown town: {goal}")
        return
    if not Reachable(start, goal):
        print("No route: the known roads do not connect these towns.")
        return
    cost, path = find_route(start, goal)
    print("Cheapest route:", " -> ".join(path))
    print(f"Total distance: {cost} km")

plan_trip(start, goal)          # expected: 351 km via Naivasha, Nakuru, Kericho
plan_trip("Nanyuki", "Kisumu")  # known town, but no roads out of it
plan_trip("Nairobi", "Mombasa") # town not in the knowledge base

# Show that a new fact changes the agent's decision
add_road("Nairobi", "Kericho", 200)
plan_trip("Nairobi", "Kisumu")     # expected: 284 km via Kericho
remove_road("Nairobi", "Kericho")
plan_trip("Nairobi", "Kisumu")     # back to 351 km


=== Trailer Route Planner ===
Trip: Nairobi -> Kisumu
Cheapest route: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu
Total distance: 351 km

=== Trailer Route Planner ===
Trip: Nanyuki -> Kisumu
No route: the known roads do not connect these towns.

=== Trailer Route Planner ===
Trip: Nairobi -> Mombasa
Unknown town: Mombasa
Added road: Nairobi -> Kericho (200 km)

=== Trailer Route Planner ===
Trip: Nairobi -> Kisumu
Cheapest route: Nairobi -> Kericho -> Kisumu
Total distance: 284 km
Removed road: Nairobi -> Kericho

=== Trailer Route Planner ===
Trip: Nairobi -> Kisumu
Cheapest route: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu
Total distance: 351 km


**Step 8: Querying the knowledge base**

In [ ]:
#ask questions to get actions based on known facts (in this case; directions)
print("Is Kericho a known town?:", Town("Kericho"))
print("Is Mombasa a known town?:", Town("Mombasa"))
print("Is there a direct road from Nakuru to Kericho?:", Road("Nakuru", "Kericho"))
print("Distance of Nakuru -> Kericho:", Distance("Nakuru", "Kericho"), "km")
print("Is Kisumu reachable from Nairobi?:", Reachable("Nairobi", "Kisumu"))
print("Is Nairobi reachable from Kisumu?:", Reachable("Kisumu", "Nairobi")) # Should be False (one-way roads)

=== Testing Predicates and Queries ===
Is Kericho a known town?: True
Is Mombasa a known town?: False
Is there a direct road from Nakuru to Kericho?: True
Distance of Nakuru -> Kericho: 107 km
Is Kisumu reachable from Nairobi?: True
Is Nairobi reachable from Kisumu?: False


**Step 9: Debugging the Knowledge Base** `[2 marks]`

In [ ]:
#testing disconnected routes, unreachable nodes, and check how our system behaves.
print("=== Debugging the Knowledge Base ===")
test_pairs = [
    ("Nairobi", "Kisumu"),
    ("Nanyuki", "Kisumu"),
    ("Nairobi", "Mombasa"),
    ("Eldoret", "Kisumu"),
    ("Kisumu", "Nairobi")
]

for start_town, end_town in test_pairs:
    print(f"\nRoute search from '{start_town}' to '{end_town}':")
    plan_trip(start_town, end_town)

=== Debugging the Knowledge Base ===

Route search from 'Nairobi' to 'Kisumu':

=== Trailer Route Planner ===
Trip: Nairobi -> Kisumu
Cheapest route: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu
Total distance: 351 km

Route search from 'Nanyuki' to 'Kisumu':

=== Trailer Route Planner ===
Trip: Nanyuki -> Kisumu
No route: the known roads do not connect these towns.

Route search from 'Nairobi' to 'Mombasa':

=== Trailer Route Planner ===
Trip: Nairobi -> Mombasa
Unknown town: Mombasa

Route search from 'Eldoret' to 'Kisumu':

=== Trailer Route Planner ===
Trip: Eldoret -> Kisumu
Cheapest route: Eldoret -> Kapsabet -> Kisumu
Total distance: 113.30000000000001 km

Route search from 'Kisumu' to 'Nairobi':

=== Trailer Route Planner ===
Trip: Kisumu -> Nairobi
No route: the known roads do not connect these towns.


### **Step 10: Final Reflection**

#### **Challenges Faced:**
1. **One-way Constraints & Graph Directionality:** The primary challenge is that highway segments in reality are often bidirectional, but our current knowledge base models links as one-way (directed). This leads to scenarios where Kisumu is reachable from Nairobi, but Nairobi is not reachable from Kisumu.
2. **Scalability of Manual Entry:** Manually encoding distances and town relations is error-prone and doesn't scale to a national highway network.
3. **Lack of Dynamic Knowledge:** Static distances do not take into account real-world road conditions like traffic jam delays, weather conditions, or construction blockades.

#### **Recommended Improvements:**
1. **Bidirectional Representation:** Implement an automatic bidirectional road generator (`add_bidirectional_road`) to streamline two-way highway routing.
2. **Integrate Real-Time State:** Expand the vocabulary to include dynamic predicates like `TrafficDelay(a, b, minutes)` or `RoadClosed(a, b)` and integrate them into the cost function of our search algorithm.
3. **API Integration:** Connect the KB to a map GIS database (e.g., OpenStreetMap) to programmatically pull actual road geometries and distances instead of manual typing.
```